# Using `mainfile.xlsx` for General Stress-Thickness Fitting

This notebook demonstrates how to drive the general fitting workflow using `mainfile.xlsx`
instead of hardcoded Python dictionaries.

## Why use mainfile.xlsx?

The `mainfile.xlsx` file lets you configure **initial parameter guesses**, **bound types**,
and **bound magnitudes** without editing any Python code.  It is the primary configuration
interface used by `fit_general_stress.py`.  Each row represents one material; columns
encode per-material or per-dataset parameters with metadata rows describing how each
parameter is bounded during optimization.

### mainfile.xlsx row layout

| Row | Content |
|-----|--------|
| 1 | Column numbers (1, 2, 3, ...) |
| 2 | Parameter names (`Sigma0`, `BetaD`, `K0`, ...) |
| 3 | Variability type: `0` = shared across all datasets of a material, `1` = per-dataset |
| 4 | Bound type: `3` = additive ±mag, `4` = multiplicative ×(1±mag), `5` = zero-lower |
| 5 | Bound magnitude |
| 6+ | One row per material — col 1 is the material name, remaining cols are initial guesses |

An optional `filename` column (vtype = 1) enables **file-based loading** from `mainfile_data/`
instead of the shared database.

**Materials covered here:** Cr, V, W (BCC refractory metals, Su data)

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

import torch
import torch.nn as nn

%matplotlib inline

# Add repo root to path
REPO_ROOT = Path('..').resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from kmorfs import (
    load_from_database,
    load_from_mainfile_data,
    GeneralSTFModel,
    read_mainfile,
    parse_mainfile_general,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1 — Inspect the raw mainfile

In [ ]:
# Paths
SCRIPT_DIR = Path('..').resolve()            # general_stress_thickness/
MAINFILE_PATH = SCRIPT_DIR / 'mainfile.xlsx'
MAINFILE_DATA_DIR = SCRIPT_DIR / 'mainfile_data'
SOURCE_CSV = REPO_ROOT / 'data' / 'source.csv'
EXPERIMENTS_CSV = REPO_ROOT / 'data' / 'all_experiments.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    matplotlib.rcParams['font.family'] = 'Times New Roman'
except Exception:
    pass

COLORS = np.array([
    '#5B9BD5', '#A5D6A7', '#F1C40F', '#E74C3C',
    '#9B59B6', '#F39C12', '#1F77B4', '#BDC3C7'
])

print(f'Mainfile path : {MAINFILE_PATH}')
print(f'Mainfile exists: {MAINFILE_PATH.exists()}')
print(f'Device        : {DEVICE}')

In [ ]:
# Display the raw mainfile DataFrame (header=None, so rows are 0-indexed)
raw_df = read_mainfile(MAINFILE_PATH)
print('Raw mainfile (first 10 rows):')
display(raw_df.head(10))

print('\nRow legend:')
print('  Row 0 : column numbers')
print('  Row 1 : parameter names')
print('  Row 2 : variability type (0=material, 1=dataset)')
print('  Row 3 : bound type (3=additive, 4=multiplicative, 5=zero-lower)')
print('  Row 4 : bound magnitude')
print('  Row 5+: material rows (col 0 = material name)')

## Step 2 — Parse the mainfile

`parse_mainfile_general()` processes the header rows and returns a dict with:

- `material_defaults` — `{material: {param: value}}` for vtype=0 (shared) params
- `process_defaults`  — `{material: {param: value}}` for vtype=1 (per-dataset) params
- `bound_types`       — `{param: int}` bound type code per parameter
- `bound_mags`        — `{param: float}` bound magnitude per parameter
- `filenames`         — list of data file names *(only present when `filename` column exists)*
- `material_names`    — parallel list of material names per filename *(file-based mode)*
- `dataset_process_defaults` — per-dataset process params *(file-based mode)*

In [ ]:
mainfile_params = parse_mainfile_general(MAINFILE_PATH)

print('Keys returned by parse_mainfile_general():')
for k, v in mainfile_params.items():
    if isinstance(v, dict):
        print(f'  {k}: dict with {len(v)} entries -> {list(v.keys())}')
    elif isinstance(v, list):
        print(f'  {k}: list of length {len(v)}')
    else:
        print(f'  {k}: {v}')

print('\nMaterial defaults (vtype=0, shared per material):')
mat_df = pd.DataFrame(mainfile_params['material_defaults']).T
display(mat_df)

print('\nProcess defaults (vtype=1, per-dataset initial guesses):')
proc_df = pd.DataFrame(mainfile_params['process_defaults']).T
display(proc_df)

In [ ]:
# Inspect bound configuration from mainfile
bound_df = pd.DataFrame({
    'bound_type': mainfile_params['bound_types'],
    'bound_mag' : mainfile_params['bound_mags'],
})
bound_df['bound_type_meaning'] = bound_df['bound_type'].map({
    3: 'additive ±mag',
    4: 'multiplicative ×(1±mag)',
    5: 'zero-lower [0, value×(1+mag)]',
})

print('Bound configuration per parameter:')
display(bound_df)

## Step 3 — Load data

The material list is read from mainfile rows, so no materials need to be hardcoded.

**Two loading modes** are supported:

| Condition | Mode | Data source |
|-----------|------|-------------|
| `filename` column absent | **Database mode** | `source.csv` + `all_experiments.csv` |
| `filename` column present | **File-based mode** | `.txt` files in `mainfile_data/` |

The cell below handles both automatically.

In [ ]:
dataset_process_defaults = None

if 'filenames' in mainfile_params:
    # ── File-based mode ──────────────────────────────────────────────────────
    # mainfile.xlsx contains a 'filename' column pointing to .txt files in
    # mainfile_data/.  This is useful when your data is not in the database.
    materials = list(mainfile_params['material_defaults'].keys())
    print(f'[File-based mode] Materials: {materials}')
    print(f'Loading {len(mainfile_params["filenames"])} datasets from {MAINFILE_DATA_DIR}')

    fit_data, process_condition, experiment_labels = load_from_mainfile_data(
        data_dir=MAINFILE_DATA_DIR,
        filenames=mainfile_params['filenames'],
        material_names=mainfile_params['material_names'],
    )
    dataset_process_defaults = mainfile_params['dataset_process_defaults']

else:
    # ── Database mode ────────────────────────────────────────────────────────
    # Material list is taken from the mainfile rows; the database is queried
    # for all available data sources for those materials.
    materials = list(mainfile_params['material_defaults'].keys())
    print(f'[Database mode] Materials: {materials}')
    print('Loading experimental data from database...')

    fit_data, process_condition, experiment_labels = load_from_database(
        source_path=SOURCE_CSV,
        experiments_path=EXPERIMENTS_CSV,
        materials=materials,
        data_sources=None,   # None = all available sources
    )

print(f'\nFound {len(experiment_labels)} experiments:')
for label in experiment_labels:
    print(f'  {label}')
print(f'\nTotal data points: {len(fit_data)}')
print('\nProcess conditions:')
display(process_condition)

## Step 4 — Build config and compare bounds

Passing `mainfile_params` to `build_config` and `setup_parameters` activates the
mainfile-specified bounds.  The cell below also shows how the mainfile bounds
compare to the hardcoded defaults.

In [ ]:
from kmorfs import compute_bounds

# Generic fallback for materials missing from the mainfile
_FALLBACK_GENERIC = {
    'SigmaC': 0, 'K0': 0, 'alpha1': 0.02, 'L0': 10, 'GrainSize_200': 15,
    'Sigma0': 10, 'BetaD': 0.05, 'Ea': 0, 'Mfda': 20, 'Di': 0.3,
    'A0': -3, 'B0': -8, 'l0': 0.8,
}

def get_material_params(material, mainfile_params):
    mat_defaults = mainfile_params.get('material_defaults', {})
    proc_defaults = mainfile_params.get('process_defaults', {})
    if material in mat_defaults or material in proc_defaults:
        params = _FALLBACK_GENERIC.copy()
        params.update(mat_defaults.get(material, {}))
        params.update(proc_defaults.get(material, {}))
        return params
    return _FALLBACK_GENERIC.copy()


def build_config(process_condition, experiment_labels, mainfile_params=None,
                 dataset_process_defaults=None):
    material_names = [label.split('_')[0] for label in experiment_labels]
    process_cols = ['SigmaC', 'K0', 'alpha1', 'L0', 'GrainSize_200']
    material_cols = ['Sigma0', 'BetaD', 'Ea', 'Mfda', 'Di', 'A0', 'B0', 'l0']
    rows = []
    seen_materials = set()
    for i, label in enumerate(experiment_labels):
        mat = material_names[i]
        params = get_material_params(mat, mainfile_params) if mainfile_params else _FALLBACK_GENERIC.copy()
        mt = process_condition.iloc[i]['Melting_T']
        row = {
            'Fit_data': label,
            'R': process_condition.iloc[i]['R'],
            'T': process_condition.iloc[i]['T'],
            'P': process_condition.iloc[i]['P'],
            'Melting_T': mt,
        }
        if dataset_process_defaults is not None:
            ds_proc = dataset_process_defaults[i]
            for col in process_cols:
                row[col] = ds_proc.get(col, params[col])
        else:
            for col in process_cols:
                row[col] = params[col]
        if mt not in seen_materials:
            for col in material_cols:
                row[col] = params[col]
            seen_materials.add(mt)
        else:
            for col in material_cols:
                row[col] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def setup_parameters(mainfile, mainfile_params=None):
    process_condition = mainfile[['R', 'T', 'P', 'Melting_T']]
    process_cols  = ['SigmaC', 'K0', 'alpha1', 'L0', 'GrainSize_200']
    material_cols = ['Sigma0', 'BetaD', 'Ea', 'Mfda', 'Di', 'A0', 'B0', 'l0']
    initial_process   = mainfile[process_cols]
    initial_materials = mainfile[material_cols].dropna()

    x_vector = np.concatenate([initial_materials.values.flatten(),
                                initial_process.values.flatten()])

    # Default bound multipliers (hardcoded fallback)
    materials_bound = np.array([2, 2, 1, 0.5, 1, 0.8, 0.8, 0.2])
    process_bound   = np.array([6, 300, 4, 1, 0.5])

    if mainfile_params:
        # ── Use bound types/magnitudes from mainfile.xlsx ──────────────────
        bound_types = mainfile_params.get('bound_types', {})
        bound_mags  = mainfile_params.get('bound_mags',  {})

        materials_lb = initial_materials.copy().astype(float)
        materials_ub = initial_materials.copy().astype(float)
        for i, col in enumerate(material_cols):
            bt = bound_types.get(col, 4)
            bm = bound_mags.get(col, materials_bound[i])
            lb, ub = compute_bounds(bt, bm, initial_materials.iloc[:, i].values)
            materials_lb.iloc[:, i] = lb
            materials_ub.iloc[:, i] = ub

        process_lb = initial_process.copy().astype(float)
        process_ub = initial_process.copy().astype(float)
        for i, col in enumerate(process_cols):
            default_bt = 5 if col != 'K0' else 3
            bt = bound_types.get(col, default_bt)
            bm = bound_mags.get(col, process_bound[i])
            lb, ub = compute_bounds(bt, bm, initial_process.iloc[:, i].values)
            process_lb.iloc[:, i] = lb
            process_ub.iloc[:, i] = ub
    else:
        # ── Hardcoded default bounds ────────────────────────────────────────
        materials_lb = initial_materials.copy().astype(float)
        materials_ub = initial_materials.copy().astype(float)
        for i in range(len(materials_bound)):
            f1 = initial_materials.iloc[:, i] * (1 / (1 + materials_bound[i]))
            f2 = initial_materials.iloc[:, i] * (1 + materials_bound[i])
            materials_lb.iloc[:, i] = np.minimum(f1, f2)
            materials_ub.iloc[:, i] = np.maximum(f1, f2)
        process_lb = initial_process.copy().astype(float)
        process_ub = initial_process.copy().astype(float)
        for i in [0, 2, 3, 4]:
            process_lb.iloc[:, i] = np.minimum(0, initial_process.iloc[:, i] * (1 + process_bound[i]))
            process_ub.iloc[:, i] = np.maximum(0, initial_process.iloc[:, i] * (1 + process_bound[i]))
        process_lb.iloc[:, 1] = initial_process.iloc[:, 1] - process_bound[1]
        process_ub.iloc[:, 1] = initial_process.iloc[:, 1] + process_bound[1]

    para_lb = np.concatenate([materials_lb.values.flatten(), process_lb.values.flatten()])
    para_ub = np.concatenate([materials_ub.values.flatten(), process_ub.values.flatten()])
    degenerate = para_lb == para_ub
    para_ub[degenerate] = para_lb[degenerate] + 1e-3
    x_vector_scaled = (x_vector - para_lb) / (para_ub - para_lb)
    n_materials = mainfile['Melting_T'].nunique()
    return {
        'x_vector_scaled': x_vector_scaled,
        'para_lb': para_lb,
        'para_ub': para_ub,
        'process_condition': process_condition,
        'initial_process': initial_process,
        'initial_materials': initial_materials,
        'process_cols': process_cols,
        'material_cols': material_cols,
        'n_materials': n_materials,
    }


# Build config and parameters using mainfile
mainfile = build_config(process_condition, experiment_labels,
                        mainfile_params, dataset_process_defaults)
params_with_mainfile = setup_parameters(mainfile, mainfile_params)
params_default       = setup_parameters(mainfile, mainfile_params=None)

material_names = [label.split('_')[0] for label in experiment_labels]
unique_materials = list(dict.fromkeys(material_names))
print(f'Unique materials : {unique_materials}')
print(f'Datasets         : {len(experiment_labels)}')

In [ ]:
# Compare lower bounds: mainfile vs hardcoded defaults for material parameters
mat_cols = params_with_mainfile['material_cols']
n_mat_params = len(mat_cols)
n_mats = params_with_mainfile['n_materials']

lb_mainfile = params_with_mainfile['para_lb'][:n_mats * n_mat_params].reshape(n_mats, n_mat_params)
ub_mainfile = params_with_mainfile['para_ub'][:n_mats * n_mat_params].reshape(n_mats, n_mat_params)
lb_default  = params_default['para_lb'][:n_mats * n_mat_params].reshape(n_mats, n_mat_params)
ub_default  = params_default['para_ub'][:n_mats * n_mat_params].reshape(n_mats, n_mat_params)

print('=== Bounds comparison (first material) ===')
cmp = pd.DataFrame({
    'param'        : mat_cols,
    'init_value'   : params_with_mainfile['initial_materials'].iloc[0].values,
    'lb_default'   : lb_default[0],
    'ub_default'   : ub_default[0],
    'lb_mainfile'  : lb_mainfile[0],
    'ub_mainfile'  : ub_mainfile[0],
    'bound_type'   : [mainfile_params['bound_types'].get(p, '?') for p in mat_cols],
    'bound_mag'    : [mainfile_params['bound_mags'].get(p, '?') for p in mat_cols],
})
display(cmp)

## Step 5 — Scale data and prepare tensors

In [ ]:
# Use mainfile-derived parameters from here on
params = params_with_mainfile

x_data = fit_data['thickness']
y_data = fit_data['StressThickness']

scaler_x    = MinMaxScaler(feature_range=(0.1, 1.1))
scaler_rawy = MinMaxScaler(feature_range=(0.1, 1.1))
scaler_fity = MinMaxScaler(feature_range=(0.1, 1.1))

x_scaled = scaler_x.fit_transform(x_data.to_numpy().reshape(-1, 1)).flatten()
y_scaled = scaler_rawy.fit_transform(y_data.to_numpy().reshape(-1, 1)).flatten()
scaler_fity.fit(y_data.to_numpy().reshape(-1, 1))

x_tensor         = torch.tensor(x_scaled, dtype=torch.float32).to(DEVICE)
y_tensor         = torch.tensor(y_scaled, dtype=torch.float32).to(DEVICE)
process_tensor   = torch.tensor(params['process_condition'].to_numpy(),
                                dtype=torch.float32).to(DEVICE)
fit_index_tensor = torch.tensor(fit_data['Index'].to_numpy(),
                                dtype=torch.long).to(DEVICE)

print(f'x_tensor shape          : {x_tensor.shape}')
print(f'y_tensor shape          : {y_tensor.shape}')
print(f'process_tensor shape    : {process_tensor.shape}')

## Step 6 — Initialize and train model

In [ ]:
n_materials = params['n_materials']

model = GeneralSTFModel(
    x0=params['x_vector_scaled'],
    para_lb=params['para_lb'],
    para_ub=params['para_ub'],
    n_materials=n_materials,
    n_process_params=len(params['process_cols']),
    n_material_params=len(params['material_cols']),
).to(DEVICE)

optimizer = torch.optim.LBFGS(
    model.parameters(), lr=0.1, max_iter=20,
    history_size=10, line_search_fn='strong_wolfe'
)
loss_fn = nn.MSELoss()

def closure():
    optimizer.zero_grad()
    y_pred = model(x_tensor, process_tensor, fit_index_tensor, scaler_x, scaler_fity)
    loss = loss_fn(y_pred, y_tensor)
    loss.backward()
    return loss

print('Training model (mainfile bounds)...')
for epoch in range(26):
    loss = optimizer.step(closure)
    if epoch % 5 == 0:
        print(f'  Epoch {epoch:3d}, Loss: {loss.item() * 100:.5f}')

## Step 7 — Evaluate and plot results

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(x_tensor, process_tensor, fit_index_tensor,
                          scaler_x, scaler_fity)

y_pred = scaler_fity.inverse_transform(
    y_pred_scaled.cpu().numpy().reshape(-1, 1)
).flatten()
fit_data = fit_data.copy()
fit_data['y_pred'] = y_pred

print(f'Prediction range  : [{y_pred.min():.3f}, {y_pred.max():.3f}] GPa·nm')
print(f'Experimental range: [{y_data.min():.3f}, {y_data.max():.3f}] GPa·nm')

In [ ]:
melting_vals = params['process_condition']['Melting_T'].unique()
n_plots = len(melting_vals)
n_cols = min(4, n_plots)
n_rows = (n_plots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
if n_plots == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)

for i, tm in enumerate(melting_vals):
    r, c = divmod(i, n_cols)
    ax = axes[r, c]
    matching_rows = params['process_condition'][
        params['process_condition']['Melting_T'] == tm
    ].index.tolist()
    color_id = 0
    for dataset_id in matching_rows:
        mask = fit_data['Index'] == dataset_id + 1
        thickness = fit_data.loc[mask, 'thickness'].reset_index(drop=True)
        raw_data  = fit_data.loc[mask, 'StressThickness'].reset_index(drop=True)
        pred_data = fit_data.loc[mask, 'y_pred'].reset_index(drop=True)
        ax.plot(thickness, pred_data, color=COLORS[color_id], linewidth=3)
        ax.scatter(thickness, raw_data,  color=COLORS[color_id], s=10)
        color_id = (color_id + 1) % len(COLORS)
    ax.set_title(f'{unique_materials[i]} (Tm = {tm:.0f} K)')
    ax.set_xlabel('Thickness (nm)')
    ax.set_ylabel('Stress·Thickness (GPa·nm)')
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    ax.tick_params(labelsize=12)

for j in range(n_plots, n_rows * n_cols):
    fig.delaxes(axes.flatten()[j])
plt.suptitle('General fitting — mainfile.xlsx bounds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 8 — Display optimized parameters

In [ ]:
with torch.no_grad():
    optimized_scaled = model.x_vector_scaled.cpu().numpy()

para_lb, para_ub = params['para_lb'], params['para_ub']
n_mat_params = len(params['material_cols'])
n_proc_params = len(params['process_cols'])

vector_param = optimized_scaled * (para_ub - para_lb) + para_lb

mat_count = n_materials * n_mat_params
materials_para = vector_param[:mat_count].reshape(n_materials, n_mat_params)
n_datasets = (len(vector_param) - mat_count) // n_proc_params
process_para = vector_param[mat_count:].reshape(n_datasets, n_proc_params)

mat_df = pd.DataFrame(materials_para, columns=params['material_cols'],
                      index=unique_materials)
print('Optimized Material Parameters:')
display(mat_df)

proc_df = pd.DataFrame(process_para, columns=params['process_cols'],
                       index=experiment_labels)
print('\nOptimized Process Parameters:')
display(proc_df)

## Summary

| Step | Function / Object |
|------|-------------------|
| Inspect raw mainfile | `read_mainfile(path)` → raw DataFrame |
| Parse mainfile       | `parse_mainfile_general(path)` → dict |
| Inspect bounds       | `mainfile_params['bound_types']`, `['bound_mags']` |
| Database loading     | `load_from_database(...)` with `materials` from mainfile |
| File-based loading   | `load_from_mainfile_data(...)` when `filename` column present |
| Apply to config      | pass `mainfile_params` to `build_config()` |
| Apply to bounds      | pass `mainfile_params` to `setup_parameters()` |

To change initial guesses or fitting bounds without editing Python:
1. Open `general_stress_thickness/mainfile.xlsx`
2. Edit initial values in row 6+ or bound magnitudes in row 5
3. Re-run `fit_general_stress.py` or this notebook